X_test = [
  [1.5],
  [5.5]
]
X_train = [
  [1],
  [2],
  [3],
  [4],
  [5]
]
y_train = [
  2,
  4,
  6,
  8,
  10
]

In [ ]:
import numpy as np


class GaussianProcessRegressor:
    def __init__(self, length_scale=1.0, sigma_f=1.0, noise=1e-6):
        self.length_scale = float(length_scale)
        self.sigma_f = float(sigma_f)
        self.noise = float(noise)
        self.X_train = None
        self.y_train = None
        self.L = None
        self.alpha = None

    def _kernel(self, X1, X2):
        X1 = np.asarray(X1, dtype=float)
        X2 = np.asarray(X2, dtype=float)
        X1_sq = np.sum(X1 ** 2, axis=1).reshape(-1, 1)
        X2_sq = np.sum(X2 ** 2, axis=1).reshape(1, -1)
        dists = X1_sq + X2_sq - 2.0 * X1 @ X2.T
        scale = -0.5 / (self.length_scale ** 2)
        return (self.sigma_f ** 2) * np.exp(scale * dists)

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y, dtype=float)
        K = self._kernel(self.X_train, self.X_train)
        K += self.noise * np.eye(self.X_train.shape[0])
        self.L = np.linalg.cholesky(K)
        v = np.linalg.solve(self.L, self.y_train)
        self.alpha = np.linalg.solve(self.L.T, v)

    def predict(self, X):
        X_test = np.asarray(X, dtype=float)
        K_star = self._kernel(self.X_train, X_test)
        mean = K_star.T @ self.alpha
        return mean.tolist()

# Instantiate model
model = GaussianProcessRegressor()
# Fit model
model.fit(X_train, y_train)
# Get predictions
y_test = model.predict(X_test)
# Output predictions
y_test